# Battery Data Evaluation

In [1]:
# Import required libraries
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import joblib
import os
import gc
from tqdm import tqdm
from typing import Dict, Any, List, Tuple
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Load the prepared battery data
battery_data_path = 'data/battery/LLmodel.pkl'
hl_data_path = 'data/battery/HLmodel.pkl'
abstraction_data_path = 'data/battery/abstraction_data.pkl'

try:
    ll_data = joblib.load(battery_data_path)
    hl_data = joblib.load(hl_data_path)
    abstraction_data = joblib.load(abstraction_data_path)
    
    print("✅ Battery data loaded successfully")
    print(f"LL data keys: {list(ll_data.keys())}")
    print(f"HL data keys: {list(hl_data.keys())}")
    print(f"Abstraction data keys: {list(abstraction_data.keys())}")
    
    # Extract key components
    Dll_samples = ll_data['data']  # Dict[Intervention, np.ndarray]
    Dhl_samples = hl_data['data']  # Dict[Intervention, np.ndarray]
    omega = abstraction_data['omega']  # Dict[Intervention, Intervention]
    
    print(f"\nAvailable interventions:")
    print(f"LL: {list(Dll_samples.keys())}")
    print(f"HL: {list(Dhl_samples.keys())}")
    print(f"\nOmega mapping: {omega}")
    
except Exception as e:
    print(f"❌ Error loading battery data: {e}")
    raise

✅ Battery data loaded successfully
LL data keys: ['graph', 'intervention_set', 'coeffs', 'intercepts', 'data', 'noise', 'node_order', 'deterministic', 'row_idx']
HL data keys: ['graph', 'intervention_set', 'coeffs', 'intercepts', 'data', 'noise', 'node_order', 'deterministic', 'row_idx']
Abstraction data keys: ['T', 'omega']

Available interventions:
LL: [None, <models.Intervention object at 0x173ec9250>, <models.Intervention object at 0x173cc0110>, <models.Intervention object at 0x173f573e0>, <models.Intervention object at 0x173f57470>]
HL: [None, <models.Intervention object at 0x173f57800>, <models.Intervention object at 0x173f577a0>, <models.Intervention object at 0x173f57740>]

Omega mapping: {None: None, <models.Intervention object at 0x1713fb0e0>: <models.Intervention object at 0x173f57380>, <models.Intervention object at 0x173f57320>: <models.Intervention object at 0x173f57d40>, <models.Intervention object at 0x173f57d10>: <models.Intervention object at 0x173f57cb0>, <models.Int

In [3]:
# --- Rebuild folds exactly like in optimization ---
def make_stratified_9010_splits(Dhl_obs: np.ndarray, test_size: float, seeds: List[int]) -> List[dict]:
    """Rebuild the same stratified splits used in optimization."""
    assert Dhl_obs.ndim == 2 and 0.0 < test_size < 1.0
    N = Dhl_obs.shape[0]
    y = Dhl_obs[:, 0]  # Use CG values for stratification
    
    label_to_indices: Dict[Any, List[int]] = {}
    for idx, lab in enumerate(y):
        label_to_indices.setdefault(lab, []).append(idx)
    
    folds = []
    for seed in seeds:
        rng = np.random.default_rng(seed)
        test_idx = []
        for _, idxs in label_to_indices.items():
            n = len(idxs)
            n_test = max(1, int(round(test_size * n)))
            chosen = rng.choice(idxs, size=n_test, replace=False)
            test_idx.append(chosen)
        test_idx = np.unique(np.concatenate(test_idx))
        train_idx = np.setdiff1d(np.arange(N), test_idx)
        folds.append({"train": train_idx, "test": test_idx, "seed": seed})
    return folds

# Build the same folds as optimization (use same seeds and test_size)
Dhl_obs_full = hl_data["data"][None]  # observational aligned (N, d_h)
folds = make_stratified_9010_splits(Dhl_obs_full, test_size=0.1, seeds=[42, 43, 44, 45, 46])
print(f"✅ Rebuilt {len(folds)} folds")
print(f"Test sizes: {[len(f['test']) for f in folds]}")
print(f"Train sizes: {[len(f['train']) for f in folds]}")

✅ Rebuilt 5 folds
Test sizes: [7, 7, 7, 7, 7]
Train sizes: [57, 57, 57, 57, 57]


In [4]:
# Load optimization results
results_dir = 'data/battery/results_empirical_9010'

try:
    # List all result files
    result_files = [f for f in os.listdir(results_dir) if f.endswith('.pkl')]
    print(f"Found {len(result_files)} result files:")
    for f in result_files:
        print(f"  - {f}")
    
    # Load and combine all method results
    all_results = {}
    
    # Load DiRoCA results
    if 'diroca_cv_results_empirical.pkl' in result_files:
        diroca_results = joblib.load(os.path.join(results_dir, 'diroca_cv_results_empirical.pkl'))
        print(f"\n✅ DiRoCA results loaded: {len(diroca_results)} epsilon configurations")
        all_results.update(diroca_results)
    
    # Load GradCA results
    if 'gradca_cv_results_empirical.pkl' in result_files:
        gradca_results = joblib.load(os.path.join(results_dir, 'gradca_cv_results_empirical.pkl'))
        print(f"✅ GradCA results loaded: {len(gradca_results)} folds")
        all_results['gradca'] = gradca_results
    
    # Load BaryCA results
    if 'baryca_cv_results_empirical.pkl' in result_files:
        baryca_results = joblib.load(os.path.join(results_dir, 'baryca_cv_results_empirical.pkl'))
        print(f"✅ BaryCA results loaded: {len(baryca_results)} folds")
        all_results['baryca'] = baryca_results
    
    # Load Abs-LiNGAM results
    if 'abslingam_cv_results_empirical.pkl' in result_files:
        abslingam_results = joblib.load(os.path.join(results_dir, 'abslingam_cv_results_empirical.pkl'))
        print(f"✅ Abs-LiNGAM results loaded: {len(abslingam_results)} folds")
        all_results['abslingam'] = abslingam_results
    
    print(f"\n✅ All results loaded successfully!")
    print(f"Combined result keys: {list(all_results.keys())}")
        
except Exception as e:
    print(f"❌ Error loading results: {e}")
    all_results = {}

Found 8 result files:
  - abslingam_cv_results_empirical.pkl
  - gradca_empirical_splits.pkl
  - gradca_cv_results_empirical.pkl
  - diroca_cv_results_empirical.pkl
  - abslingam_empirical_splits.pkl
  - diroca_empirical_splits.pkl
  - baryca_empirical_splits.pkl
  - baryca_cv_results_empirical.pkl

✅ DiRoCA results loaded: 5 epsilon configurations
✅ GradCA results loaded: 5 folds
✅ BaryCA results loaded: 5 folds
✅ Abs-LiNGAM results loaded: 5 folds

✅ All results loaded successfully!
Combined result keys: ['epsilon_0.558_delta_0.417', 'epsilon_1.0_delta_1.0', 'epsilon_2.0_delta_2.0', 'epsilon_4.0_delta_4.0', 'epsilon_8.0_delta_8.0', 'gradca', 'baryca', 'abslingam']


In [5]:
# --- T Matrix Orientation Helper ---
def coerce_T(T, d_h, d_l):
    """Ensure T matrix has correct orientation (d_h, d_l)."""
    T = np.asarray(T)
    if T.shape == (d_h, d_l): 
        return T
    if T.shape == (d_l, d_h): 
        return T.T
    raise ValueError(f"Unexpected T shape {T.shape}, expected {(d_h, d_l)} or {(d_l, d_h)}")

# Get dimensions for T matrix validation
d_h = hl_data["data"][None].shape[1]  # HL features
d_l = ll_data["data"][None].shape[1]  # LL features
print(f"Expected T matrix shape: ({d_h}, {d_l})")
print(f"HL features: {d_h}, LL features: {d_l}")

Expected T matrix shape: (2, 3)
HL features: 2, LL features: 3


In [6]:
# Contamination function for battery data (similar to CMNIST but for tabular data)
def apply_huber_contamination_battery(clean_data, alpha, noise_scale, noise_dims, seed=None, loc=0.0):
    """Contaminates specific dimensions of battery data with zero-mean noise.
    
    Args:
        clean_data: np.ndarray or torch.Tensor of shape (n_samples, n_features)
        alpha: contamination fraction (0.0 = no contamination, 1.0 = full contamination)
        noise_scale: standard deviation of noise
        noise_dims: indices or slice of dimensions to contaminate
        seed: random seed for reproducibility
        loc: mean of noise distribution (default 0.0 for zero-mean)
    
    Returns:
        contaminated_data: same type as input
    """
    if alpha == 0 or noise_scale == 0:
        return clean_data if isinstance(clean_data, torch.Tensor) else torch.tensor(clean_data, dtype=torch.float32)

    data_tensor = clean_data if isinstance(clean_data, torch.Tensor) else torch.tensor(clean_data, dtype=torch.float32)
    data_cont = data_tensor.clone().to(torch.float32)
    data_to_noise = data_cont[:, noise_dims].data

    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=loc, scale=noise_scale, size=data_to_noise.shape).astype(np.float32)
    noise_tensor = torch.tensor(noise, dtype=torch.float32, device=data_tensor.device)
    noisy_slice = data_to_noise + noise_tensor

    if alpha >= 1.0:
        data_cont[:, noise_dims] = noisy_slice
        return data_cont

    n_samples = data_tensor.shape[0]
    n_contaminate = int(alpha * n_samples)
    if n_contaminate == 0: 
        return data_cont

    idx_to_contaminate = rng.choice(n_samples, size=n_contaminate, replace=False)
    data_cont[idx_to_contaminate, noise_dims] = noisy_slice[idx_to_contaminate]
    return data_cont

print("✅ Contamination function defined for battery data.")

✅ Contamination function defined for battery data.


In [7]:
# Error calculation function for battery data (FIXED normalization)
def calculate_empirical_error_battery(T_matrix, Dll_test, Dhl_test):
    """Calculates abstraction error for the linear T matrix on battery data.
    
    Args:
        T_matrix: transformation matrix (d_h, d_l)
        Dll_test: low-level test data (n_test, d_l)
        Dhl_test: high-level test data (n_test, d_h)
    
    Returns:
        error: Frobenius norm squared error per sample per dimension
    """
    try:
        T_matrix = T_matrix if isinstance(T_matrix, torch.Tensor) else torch.tensor(T_matrix, dtype=torch.float32)
        Dll_test = Dll_test if isinstance(Dll_test, torch.Tensor) else torch.tensor(Dll_test, dtype=torch.float32)
        Dhl_test = Dhl_test if isinstance(Dhl_test, torch.Tensor) else torch.tensor(Dhl_test, dtype=torch.float32)
        
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        T_matrix = T_matrix.to(device)
        Dll_test = Dll_test.to(device)
        Dhl_test = Dhl_test.to(device)

        # Check dimension compatibility
        if T_matrix.shape[1] != Dll_test.shape[1]: 
            print(f"Dimension mismatch T vs LL: T={T_matrix.shape}, LL={Dll_test.shape}")
            return float('inf')
        if T_matrix.shape[0] != Dhl_test.shape[1]: 
            print(f"Dimension mismatch T vs HL: T={T_matrix.shape}, HL={Dhl_test.shape}")
            return float('inf')

        with torch.no_grad():
            Dhl_pred = Dll_test @ T_matrix.T  # (n_test, d_l) @ (d_l, d_h) = (n_test, d_h)
            diff = Dhl_pred - Dhl_test
            # FIXED: Use same normalization as optimization loop
            err = torch.norm(diff, p='fro')**2 / (diff.shape[0] * diff.shape[1])
        return float(err.item())

    except Exception as e:
        print(f"Error in calculate_empirical_error_battery: {e}")
        return float('inf')

print("✅ Error calculation function defined for battery data.")

✅ Error calculation function defined for battery data.


In [8]:
# Evaluation parameters
N_TRIALS = 5
NOISE_SCALE_FOR_ALPHA1 = 5.0  # Standard deviation for full contamination
ALPHA_VALUES_TO_TEST = [0.0, 1.0]  # Clean data and fully contaminated data

# Define which dimensions to contaminate
# For battery data, we contaminate all features except the first one (which is CG)
sample_ll = next(iter(Dll_samples.values()))
sample_hl = next(iter(Dhl_samples.values()))
n_features_ll = sample_ll.shape[1]
n_features_hl = sample_hl.shape[1]
LL_NOISE_DIMS = slice(1, n_features_ll)  # Contaminate all features except first (CG)
HL_NOISE_DIMS = slice(1, n_features_hl)  # Contaminate all features except first (CG)

print(f"LL data shape: {sample_ll.shape}")
print(f"HL data shape: {sample_hl.shape}")
print(f"Will contaminate LL dimensions: {LL_NOISE_DIMS}")
print(f"Will contaminate HL dimensions: {HL_NOISE_DIMS}")

print(f"\nEvaluation parameters:")
print(f"  - N_TRIALS: {N_TRIALS}")
print(f"  - NOISE_SCALE_FOR_ALPHA1: {NOISE_SCALE_FOR_ALPHA1}")
print(f"  - ALPHA_VALUES_TO_TEST: {ALPHA_VALUES_TO_TEST}")

LL data shape: (64, 3)
HL data shape: (64, 2)
Will contaminate LL dimensions: slice(1, 3, None)
Will contaminate HL dimensions: slice(1, 2, None)

Evaluation parameters:
  - N_TRIALS: 5
  - NOISE_SCALE_FOR_ALPHA1: 5.0
  - ALPHA_VALUES_TO_TEST: [0.0, 1.0]


In [9]:
# Set up output directory
output_dir = 'data/battery/evaluation_results'
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

print("\n--- Starting Battery Data Evaluation (Corrected) ---")

evaluation_records = []

# Count total configurations for progress bar
total_configs = 0
for method_group_key, method_results_inner in all_results.items():
    if not isinstance(method_results_inner, dict): continue
    for fold_key, fold_data in method_results_inner.items():
        if not fold_key.startswith('fold_'): continue
        if method_group_key.startswith('epsilon_'):  # DiRoCA
            total_configs += 1
        elif method_group_key in ['gradca', 'baryca']:  # GradCA, BaryCA
            total_configs += 1
        elif method_group_key == 'abslingam':  # Abs-LiNGAM
            total_configs += len(fold_data)  # Perfect + Noisy

total_configs *= len(ALPHA_VALUES_TO_TEST) * N_TRIALS
print(f"Total evaluation configurations: {total_configs}")

if total_configs == 0:
    print("❌ No valid training results found. Cannot run evaluation.")
    print("--- Battery Data Evaluation Complete ---")
else:
    print("✅ Found valid training results. Starting evaluation...")
    
    pbar_eval = tqdm(total=total_configs, desc="Evaluating Battery Methods")
    
    # Evaluation loop for all methods
    for method_group_key, method_results_inner in all_results.items():
        if not isinstance(method_results_inner, dict): continue
        
        for fold_key, fold_data in method_results_inner.items():
            if not fold_key.startswith('fold_'): continue
            
            # Get fold index and global test indices
            i = int(fold_key.split('_')[-1])
            global_test_idx = set(folds[i]["test"])
            
            if method_group_key.startswith('epsilon_'):  # DiRoCA
                run_result = fold_data
                run_key = method_group_key
                eval_method_name = f"DiRoCA ({run_key})"
                
                if 'T' not in run_result: 
                    pbar_eval.update(len(ALPHA_VALUES_TO_TEST) * N_TRIALS)
                    continue
                
                # FIXED: Coerce T matrix orientation
                T_matrix = coerce_T(run_result['T'], d_h, d_l)
                
                # Inner evaluation loop
                for alpha in ALPHA_VALUES_TO_TEST:
                    noise_scale = 0.0 if np.isclose(alpha, 0.0) else NOISE_SCALE_FOR_ALPHA1
                    
                    for trial in range(N_TRIALS):
                        trial_errors = []
                        
                        for iota, eta in list(omega.items()):
                            try:
                                if iota not in Dll_samples or eta not in Dhl_samples:
                                    continue
                                
                                # FIXED: Use proper test indices and alignment
                                ll_bucket_idx = ll_data["row_idx"].get(iota, None)
                                hl_bucket_idx = hl_data["row_idx"].get(eta, None)
                                if ll_bucket_idx is None or hl_bucket_idx is None:
                                    continue
                                
                                # Restrict to test split
                                ll_test_mask = np.array([g in global_test_idx for g in ll_bucket_idx], dtype=bool)
                                hl_test_mask = np.array([g in global_test_idx for g in hl_bucket_idx], dtype=bool)
                                if not ll_test_mask.any() or not hl_test_mask.any():
                                    continue
                                
                                # Take raw rows
                                Dll_test_clean = Dll_samples[iota][ll_test_mask]
                                Dhl_test_clean = Dhl_samples[eta][hl_test_mask]
                                
                                # Align LL/HL by common global ids (robust alignment)
                                ll_ids = ll_bucket_idx[ll_test_mask]
                                hl_ids = hl_bucket_idx[hl_test_mask]
                                common = np.intersect1d(ll_ids, hl_ids)
                                if common.size == 0:
                                    continue
                                    
                                pos_l = {g: p for p, g in enumerate(ll_ids)}
                                pos_h = {g: p for p, g in enumerate(hl_ids)}
                                sel_l = np.array([pos_l[g] for g in common], dtype=int)
                                sel_h = np.array([pos_h[g] for g in common], dtype=int)
                                
                                Dll_test_clean = Dll_test_clean[sel_l]
                                Dhl_test_clean = Dhl_test_clean[sel_h]
                                
                                # Apply contamination
                                seed = hash((i, run_key, float(alpha), float(noise_scale), trial, str(iota))) % (2**32)
                                Dll_test_cont = apply_huber_contamination_battery(
                                    Dll_test_clean, alpha, noise_scale, LL_NOISE_DIMS, seed=seed, loc=0.0
                                )
                                Dhl_test_cont = apply_huber_contamination_battery(
                                    Dhl_test_clean, alpha, noise_scale, HL_NOISE_DIMS, seed=seed, loc=0.0
                                )
                                
                                # Calculate error
                                error = calculate_empirical_error_battery(T_matrix, Dll_test_cont, Dhl_test_cont)
                                if not np.isnan(error) and error != float('inf'):
                                    trial_errors.append(error)
                                    
                            except Exception as e:
                                print(f"ERROR in evaluation: {e} | Context: M={eval_method_name}, F={i}, R={run_key}, A={alpha}, T={trial}, Iota={iota}")
                                trial_errors.append(np.nan)

                        # Record results
                        record = {
                            'method': eval_method_name,
                            'fold': i,
                            'alpha': float(alpha),
                            'noise_scale': float(noise_scale),
                            'trial': trial,
                            'error': float(np.nanmean(trial_errors)) if trial_errors else np.nan
                        }
                        
                        if 'epsilon' in run_result:
                            record['train_epsilon'] = run_result.get('epsilon', np.nan)
                            record['train_delta'] = run_result.get('delta', np.nan)
                        
                        evaluation_records.append(record)
                        pbar_eval.update(1)
                        
            elif method_group_key in ['gradca', 'baryca']:  # GradCA, BaryCA
                run_result = fold_data
                run_key = f"{method_group_key}_run"
                eval_method_name = method_group_key.title()
                
                if 'T' not in run_result: 
                    pbar_eval.update(len(ALPHA_VALUES_TO_TEST) * N_TRIALS)
                    continue
                
                # FIXED: Coerce T matrix orientation
                T_matrix = coerce_T(run_result['T'], d_h, d_l)
                
                # Inner evaluation loop (same as DiRoCA)
                for alpha in ALPHA_VALUES_TO_TEST:
                    noise_scale = 0.0 if np.isclose(alpha, 0.0) else NOISE_SCALE_FOR_ALPHA1
                    
                    for trial in range(N_TRIALS):
                        trial_errors = []
                        
                        for iota, eta in list(omega.items()):
                            try:
                                if iota not in Dll_samples or eta not in Dhl_samples:
                                    continue
                                
                                # FIXED: Use proper test indices and alignment
                                ll_bucket_idx = ll_data["row_idx"].get(iota, None)
                                hl_bucket_idx = hl_data["row_idx"].get(eta, None)
                                if ll_bucket_idx is None or hl_bucket_idx is None:
                                    continue
                                
                                # Restrict to test split
                                ll_test_mask = np.array([g in global_test_idx for g in ll_bucket_idx], dtype=bool)
                                hl_test_mask = np.array([g in global_test_idx for g in hl_bucket_idx], dtype=bool)
                                if not ll_test_mask.any() or not hl_test_mask.any():
                                    continue
                                
                                # Take raw rows
                                Dll_test_clean = Dll_samples[iota][ll_test_mask]
                                Dhl_test_clean = Dhl_samples[eta][hl_test_mask]
                                
                                # Align LL/HL by common global ids (robust alignment)
                                ll_ids = ll_bucket_idx[ll_test_mask]
                                hl_ids = hl_bucket_idx[hl_test_mask]
                                common = np.intersect1d(ll_ids, hl_ids)
                                if common.size == 0:
                                    continue
                                    
                                pos_l = {g: p for p, g in enumerate(ll_ids)}
                                pos_h = {g: p for p, g in enumerate(hl_ids)}
                                sel_l = np.array([pos_l[g] for g in common], dtype=int)
                                sel_h = np.array([pos_h[g] for g in common], dtype=int)
                                
                                Dll_test_clean = Dll_test_clean[sel_l]
                                Dhl_test_clean = Dhl_test_clean[sel_h]
                                
                                # Apply contamination
                                seed = hash((i, run_key, float(alpha), float(noise_scale), trial, str(iota))) % (2**32)
                                Dll_test_cont = apply_huber_contamination_battery(
                                    Dll_test_clean, alpha, noise_scale, LL_NOISE_DIMS, seed=seed, loc=0.0
                                )
                                Dhl_test_cont = apply_huber_contamination_battery(
                                    Dhl_test_clean, alpha, noise_scale, HL_NOISE_DIMS, seed=seed, loc=0.0
                                )
                                
                                # Calculate error
                                error = calculate_empirical_error_battery(T_matrix, Dll_test_cont, Dhl_test_cont)
                                if not np.isnan(error) and error != float('inf'):
                                    trial_errors.append(error)
                                    
                            except Exception as e:
                                print(f"ERROR in evaluation: {e} | Context: M={eval_method_name}, F={i}, R={run_key}, A={alpha}, T={trial}, Iota={iota}")
                                trial_errors.append(np.nan)

                        # Record results
                        record = {
                            'method': eval_method_name,
                            'fold': i,
                            'alpha': float(alpha),
                            'noise_scale': float(noise_scale),
                            'trial': trial,
                            'error': float(np.nanmean(trial_errors)) if trial_errors else np.nan
                        }
                        
                        evaluation_records.append(record)
                        pbar_eval.update(1)
                        
            elif method_group_key == 'abslingam':  # Abs-LiNGAM
                # For Abs-LiNGAM, get train_N and test_N from the first style
                first_style = list(fold_data.keys())[0]
                
                for run_key, run_result in fold_data.items():
                    if 'T' not in run_result: 
                        pbar_eval.update(len(ALPHA_VALUES_TO_TEST) * N_TRIALS)
                        continue
                    
                    eval_method_name = f"Abs-LiNGAM ({run_key})"
                    # FIXED: Coerce T matrix orientation
                    T_matrix = coerce_T(run_result['T'], d_h, d_l)
                    
                    # Inner evaluation loop (same as others)
                    for alpha in ALPHA_VALUES_TO_TEST:
                        noise_scale = 0.0 if np.isclose(alpha, 0.0) else NOISE_SCALE_FOR_ALPHA1
                        
                        for trial in range(N_TRIALS):
                            trial_errors = []
                            
                            for iota, eta in list(omega.items()):
                                try:
                                    if iota not in Dll_samples or eta not in Dhl_samples:
                                        continue
                                    
                                    # FIXED: Use proper test indices and alignment
                                    ll_bucket_idx = ll_data["row_idx"].get(iota, None)
                                    hl_bucket_idx = hl_data["row_idx"].get(eta, None)
                                    if ll_bucket_idx is None or hl_bucket_idx is None:
                                        continue
                                    
                                    # Restrict to test split
                                    ll_test_mask = np.array([g in global_test_idx for g in ll_bucket_idx], dtype=bool)
                                    hl_test_mask = np.array([g in global_test_idx for g in hl_bucket_idx], dtype=bool)
                                    if not ll_test_mask.any() or not hl_test_mask.any():
                                        continue
                                    
                                    # Take raw rows
                                    Dll_test_clean = Dll_samples[iota][ll_test_mask]
                                    Dhl_test_clean = Dhl_samples[eta][hl_test_mask]
                                    
                                    # Align LL/HL by common global ids (robust alignment)
                                    ll_ids = ll_bucket_idx[ll_test_mask]
                                    hl_ids = hl_bucket_idx[hl_test_mask]
                                    common = np.intersect1d(ll_ids, hl_ids)
                                    if common.size == 0:
                                        continue
                                        
                                    pos_l = {g: p for p, g in enumerate(ll_ids)}
                                    pos_h = {g: p for p, g in enumerate(hl_ids)}
                                    sel_l = np.array([pos_l[g] for g in common], dtype=int)
                                    sel_h = np.array([pos_h[g] for g in common], dtype=int)
                                    
                                    Dll_test_clean = Dll_test_clean[sel_l]
                                    Dhl_test_clean = Dhl_test_clean[sel_h]
                                    
                                    # Apply contamination
                                    seed = hash((i, run_key, float(alpha), float(noise_scale), trial, str(iota))) % (2**32)
                                    Dll_test_cont = apply_huber_contamination_battery(
                                        Dll_test_clean, alpha, noise_scale, LL_NOISE_DIMS, seed=seed, loc=0.0
                                    )
                                    Dhl_test_cont = apply_huber_contamination_battery(
                                        Dhl_test_clean, alpha, noise_scale, HL_NOISE_DIMS, seed=seed, loc=0.0
                                    )
                                    
                                    # Calculate error
                                    error = calculate_empirical_error_battery(T_matrix, Dll_test_cont, Dhl_test_cont)
                                    if not np.isnan(error) and error != float('inf'):
                                        trial_errors.append(error)
                                        
                                except Exception as e:
                                    print(f"ERROR in evaluation: {e} | Context: M={eval_method_name}, F={i}, R={run_key}, A={alpha}, T={trial}, Iota={iota}")
                                    trial_errors.append(np.nan)

                            # Record results
                            record = {
                                'method': eval_method_name,
                                'fold': i,
                                'alpha': float(alpha),
                                'noise_scale': float(noise_scale),
                                'trial': trial,
                                'error': float(np.nanmean(trial_errors)) if trial_errors else np.nan
                            }
                            
                            evaluation_records.append(record)
                            pbar_eval.update(1)

    pbar_eval.close()
    
    # Save results
    full_results_df = pd.DataFrame(evaluation_records)
    eval_output_path = os.path.join(output_dir, "battery_evaluation_results_corrected.pkl")
    full_results_df.to_pickle(eval_output_path)
    print(f"\n✅ Evaluation results saved to {eval_output_path}")
    print(f"Total records: {len(evaluation_records)}")
    print(f"Methods found: {sorted(full_results_df['method'].unique())}")

print("\n--- Battery Data Evaluation Complete ---")

Output directory: data/battery/evaluation_results

--- Starting Battery Data Evaluation (Corrected) ---
Total evaluation configurations: 450
✅ Found valid training results. Starting evaluation...


Evaluating Battery Methods: 100%|██████████| 450/450 [00:00<00:00, 1069.08it/s]


✅ Evaluation results saved to data/battery/evaluation_results/battery_evaluation_results_corrected.pkl
Total records: 450
Methods found: ['Abs-LiNGAM (Noisy)', 'Abs-LiNGAM (Perfect)', 'Baryca', 'DiRoCA (epsilon_0.558_delta_0.417)', 'DiRoCA (epsilon_1.0_delta_1.0)', 'DiRoCA (epsilon_2.0_delta_2.0)', 'DiRoCA (epsilon_4.0_delta_4.0)', 'DiRoCA (epsilon_8.0_delta_8.0)', 'Gradca']

--- Battery Data Evaluation Complete ---


In [10]:
# Load and analyze the corrected results
if 'full_results_df' in locals() and not full_results_df.empty:
    print("\n--- Analyzing Corrected Battery Evaluation Results ---")
    
    # Set pandas display options
    pd.set_option('display.max_rows', None)
    pd.set_option('display.width', 1000)
    
    # Results on clean data (α = 0.0)
    df_clean = full_results_df[np.isclose(full_results_df['alpha'], 0.0)]
    if not df_clean.empty:
        summary_clean = df_clean.groupby('method')['error'].agg(['mean', 'std', 'count']).sort_values('mean')
        print("\n--- Results on Clean Data (α = 0.0, σ = 0.0) ---")
        print(summary_clean)
    
    # Results on fully contaminated data (α = 1.0)
    df_noisy = full_results_df[np.isclose(full_results_df['alpha'], 1.0)]
    if not df_noisy.empty:
        summary_noisy = df_noisy.groupby('method')['error'].agg(['mean', 'std', 'count']).sort_values('mean')
        print(f"\n--- Results on Fully Contaminated Data (α = 1.0, σ = {NOISE_SCALE_FOR_ALPHA1:.1f}) ---")
        print(summary_noisy)
    
    # Robustness analysis (difference between clean and noisy)
    if not df_clean.empty and not df_noisy.empty:
        clean_means = df_clean.groupby('method')['error'].mean()
        noisy_means = df_noisy.groupby('method')['error'].mean()
        robustness = noisy_means - clean_means
        robustness_df = pd.DataFrame({
            'clean_error': clean_means,
            'noisy_error': noisy_means,
            'robustness_degradation': robustness
        }).sort_values('robustness_degradation')
        
        print("\n--- Robustness Analysis (Noisy - Clean Error) ---")
        print("Lower values indicate better robustness:")
        print(robustness_df)
    
    # Reset pandas options
    pd.reset_option('display.max_rows')
    pd.reset_option('display.width')
    
else:
    print("❌ No evaluation results to analyze.")


--- Analyzing Corrected Battery Evaluation Results ---

--- Results on Clean Data (α = 0.0, σ = 0.0) ---
                                           mean         std  count
method                                                            
Abs-LiNGAM (Perfect)                  58.851008   11.447126     25
DiRoCA (epsilon_1.0_delta_1.0)        77.799464   20.238161     25
DiRoCA (epsilon_0.558_delta_0.417)    83.962193   29.080132     25
DiRoCA (epsilon_4.0_delta_4.0)        91.758845   27.507168     25
DiRoCA (epsilon_8.0_delta_8.0)        91.758845   27.507168     25
DiRoCA (epsilon_2.0_delta_2.0)        92.550615   26.514923     25
Gradca                                93.528501   26.836730     25
Abs-LiNGAM (Noisy)                   352.822360   47.801276     25
Baryca                              2824.974383  488.986152     25

--- Results on Fully Contaminated Data (α = 1.0, σ = 5.0) ---
                                           mean          std  count
method                    